# Two domains, one model, no rehearsal

This notebook adapts **Qwen3-0.6B** to one domain, then to a second one, and measures at every
step what was learned and what was kept. Between the two domains nothing from the first is
stored, replayed or revisited: the only thing that carries the first domain forward is a
distribution of hidden states.

Put your own corpus where the two books are and you have the whole flow. The habit worth taking
from it is the **control**: two of the runs below have the anchor switched off — same corpus,
same optimizer steps, same everything else — and the difference between the anchored and the
unanchored column is what the method bought.

## What happens

| | |
|---|---|
| 1 | Download, build or reuse the self-generated `p(h)` artifact and look inside it |
| 2 | Download two public-domain books, one per domain |
| 3 | Make a workspace |
| 4 | **Stage 1** — adapt to domain A |
| 5 | **The control** — the same stage with the anchor off, read on both axes |
| 6 | `extend` — fold A into the model *and* into `p(h)` |
| 7 | **Stage 2** — adapt to domain B, anchored against the extended `p(h)` |
| 8 | **The control again** — stage 2 with the anchor off, and what happened to A |
| 9 | Export a plain checkpoint and generate from it |

## What you need

One CUDA card. An unanchored 4-epoch run on this Darwin text, like this notebook's controls, read
up to 13,971 MiB (about 13.6 GiB) by `nvidia-smi` on an RTX 3090, an upper bound on what it needs
(`docs/faq.md`). On a cold store, cell 4 downloads the published artifact for this model; building
one for a model with none published used about 13 GB for its fit on an RTX 3090. The package,
installed from its repository as the README says: `git clone
https://github.com/sparkdoc/lfa-anchoring`, then `pip install -c constraints-tested.txt -e .`
inside it (these outputs were recorded at version 0.2.0). And network for the model, the two books
and WikiText-2. **No API keys**: every number is a perplexity computed on your own machine.

## What it costs, and what the real recipe costs

The settings in the next cell are **demo settings**, chosen so the notebook finishes in about
thirty-five minutes on one RTX 3090 once the artifact is in the store. They are not the
recommended operating point.

| | this notebook | the shipped recipe (`lfa/recipes/qwen3-0.6b.yaml`) |
|---|---|---|
| text per domain | 0.97 MB and 0.48 MB | — (its λ was calibrated on this Darwin text) |
| epochs | 4 | 15 |
| wall clock per anchored stage | 12.8 and 7.7 min, each including writing its supplement | about 25 min of training on this Darwin text |
| wall clock per evaluation and its control | 7.2 and 4.3 min | — |
| training runs | 4 (two stages, two controls) | 1 per stage |
| whole notebook | 35.5 min, with the artifact already in the store | — |

Everything else — rank 32, λ = 1,000,000, μ = 0.05, the layer schedule, the optimizer — is the
shipped recipe untouched, because λ is coupled to the rank and to the artifact and porting it is
a different regularizer (`docs/concepts.md`). Only the run length changes here.

This is **one run per cell, one seed, no error bars**: read the direction and size of each
contrast, not the digits. The outputs were recorded on 2026-10-06 on one RTX 3090, in the
working directory `/tmp/lfa-nb-run/cwd`. The other filesystem paths in them are shortened: the
home directory to `~`, and the directories above this checkout to `…`. No number is altered.


**Recorded outputs.** The outputs below were recorded on 2026-10-06 on one NVIDIA GeForce RTX 3090
(24 GB), with lfa 0.2.0 and torch 2.10.0+cu128. The artifact is the self-generated one at the
recorded frame — 2,500 documents × 2,048 tokens, fit at 600k samples per site, K = 32 — already in
the store, and byte for byte the file `lfa init` downloads for Qwen3-0.6B
(`lfa/artifact/published.json`). The supplement is on: each stage mixes in question-and-answer
pairs the model wrote over its corpus. A run on another machine can differ, and so can one over an
artifact built there rather than downloaded; the prose around each table says what to look for,
not which number to expect.

In [1]:
import os
from pathlib import Path

# --- demo scale (see the table above) ---------------------------------------------------------
EPOCHS = 4              # demo wall clock; the shipped recipe trains 15 (see section 4)
N_WINDOWS = 200         # WikiText-2 sliding windows for the general axis (200 x 512 tokens)
EXTEND_NEED = 20_000    # hidden states collected per site when folding domain A into p(h)
EXTEND_K = 4            # mixture components fitted per site for the new domain

DEVICE = "cuda:0"
MODEL_ID = "Qwen/Qwen3-0.6B"
# a path to an artifact this package built skips the build; otherwise it is built once and reused
ARTIFACT = os.environ.get("LFA_ARTIFACT", "self-generated")

DEMO = Path("lfa_demo").resolve()      # everything this notebook creates lives here
DATA, WS = DEMO / "data", DEMO / "workspace"
DEMO.mkdir(exist_ok=True)
print(DEMO)

import logging, os, sys, time, textwrap

logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout, force=True)
logging.getLogger("datasets").setLevel(logging.WARNING)

import torch
import lfa

NOTEBOOK_STARTED = time.time()
print(f"lfa          {lfa.__version__}")
print(f"torch        {torch.__version__}")
print(f"CUDA device  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
assert torch.cuda.is_available(), "this notebook needs a CUDA card"

/tmp/lfa-nb-run/cwd/lfa_demo


…/lfa-anchoring/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


lfa          0.2.0
torch        2.10.0+cu128
CUDA device  NVIDIA GeForce RTX 3090


## 1. The artifact: what makes adaptation data-free

LFA's preservation signal does not come from text. It comes from a file, built **once per
model**, recording what the hidden states arriving at each anchored sub-module look like: a mean,
a PCA basis with its eigenvalues, and a small mixture fitted in that basis, per site. The
training loop samples hidden states from it instead of reading a corpus.

The file is built from the model's own text: the model writes 2,500 documents, each started
from its bare document-start token, and the statistics are fitted on them. It is kept in the
local store (`~/.cache/lfa/artifacts`, or `$LFA_ARTIFACT_STORE`). For a bundled model the first
run downloads the published one (132 MB with its corpus for Qwen3-0.6B); any other model builds
it, which costs hours (about 3 h 51 min for Qwen3-0.6B on one RTX 3090). Every later run reuses
it; the cell lists what the store holds. Set `LFA_ARTIFACT` to an
artifact this package built (for example a stored `artifact.pt`) to skip the build.

The cell then opens the artifact, because "not the old data in disguise" is checkable: tensors of
statistics, counts, floats, and one string per site naming the mixture fitted. No token ids, no
documents, nothing sequence-shaped.


In [2]:
from lfa.artifact.store import list_store, obtain_self_generated
from lfa.selfgen.artifact_corpus import SelfGenOptions

t0 = time.time()
if ARTIFACT == "self-generated":
    artifact_path, _ = obtain_self_generated(MODEL_ID, SelfGenOptions())
else:
    artifact_path = Path(ARTIFACT)
print(f"{artifact_path}  ({artifact_path.stat().st_size / 1e6:.1f} MB, {time.time()-t0:.0f}s)")
for entry in list_store():
    print(f"   store: {entry['model_id']}  {entry['state']}  {entry['path']}")

from lfa.artifact.schema import load_artifact

params = load_artifact(artifact_path)      # int8 on disk, dequantized on load
meta = params["__meta__"]
sites = [k for k in params if k != "__meta__"]

print("__meta__:")
for key, value in meta.items():
    print(f"   {key:<18} {value if not isinstance(value, list) else value}")
print(f"\n{len(sites)} site blocks, e.g. {sites[:3]} ... {sites[-2:]}")

example = params["12_pre_mlp"]
print("\nwhat one site stores (layer 12, the input of the whole MLP):")
for key, value in example.items():
    shape = tuple(value.shape) if torch.is_tensor(value) else value
    print(f"   {key:<22} {shape}")

types = {type(v).__name__ for block in sites for v in params[block].values()}
print(f"\nvalue types across every site block: {sorted(types)}")

Reused the self-generated artifact built 2026-10-04 from ~/.cache/lfa/artifacts/Qwen--Qwen3-0.6B-a15c53884fd5-9b4469a9df3a


~/.cache/lfa/artifacts/Qwen--Qwen3-0.6B-a15c53884fd5-9b4469a9df3a/artifact.pt  (110.0 MB, 1s)
   store: Qwen/Qwen3-0.6B  built  ~/.cache/lfa/artifacts/Qwen--Qwen3-0.6B-a15c53884fd5-9b4469a9df3a
   store: Qwen/Qwen3-1.7B  built  ~/.cache/lfa/artifacts/Qwen--Qwen3-1.7B-0ca2680d68c1-9b4469a9df3a
__meta__:
   model_id           Qwen/Qwen3-0.6B
   hidden_size        1024
   num_layers         28
   sites              ['pre_qkv', 'pre_o', 'pre_mlp', 'pre_lm_head']
   n_samples_total    603008
   built_with         lfa-anchoring
   format_version     1
   lfa_version        0.2.0
   provenance         self-generated
   corpus_sha256      b9cfe7d681e7498ff37b3c30882514963bdf1d439fb173e4c986906aba208710
   layer_group_size   28
   selfgen_frame      {'n_raw': 2500, 'n_chat': 0, 'max_new_tokens': 2048, 'seed': 42, 'chat_seed': 43, 'min_chars': 1, 'max_repeat_ratio': 1.0, 'burn_in_tokens': 0, 'max_samples': 600000, 'gmm_k': 32, 'pca_variance': 0.95, 'reservoir_size': 200000}

85 site blocks, e.g.

## 2. Two domains, from Project Gutenberg

Two public-domain books, far enough apart — different century, different purpose, different
vocabulary — that "adapted to A" and "adapted to B" are visibly different states of the same
model, and neither is WikiText-2, which is the general axis.

| | domain A | domain B |
|---|---|---|
| book | *On the Origin of Species* | *Domestic Cookery, Useful Receipts, and Hints to Young Housekeepers* |
| author, year | Charles Darwin, 1859 | Elizabeth E. Lea, 1853 |
| Gutenberg id | 1228 | 9101 |
| download | `https://www.gutenberg.org/cache/epub/1228/pg1228.txt` (0.97 MB) | `https://www.gutenberg.org/cache/epub/9101/pg9101.txt` (0.48 MB) |

The next cell downloads them, strips the Gutenberg licence header and footer, and accumulates
paragraphs into documents of about 3,500 characters — roughly 850 tokens, so most documents are
one or two of the 512-token chunks the trainer reads. **This is the cell to replace with your own
corpus**: everything after it wants a flat directory of `.txt` files, which `lfa prepare-domain`
will build for you from text, Markdown, HTML or PDF; for one long file, its `--split-chars 3500`
splits it into documents the way this cell does.

Every eighth document goes to `heldout/` and is **never trained on, at either stage**. Every
domain perplexity below is measured on those documents, so all of them are comparable with each
other. (The trainer separately keeps a tenth of the *training* documents back for its own
per-epoch curve — a different split, used only to choose the number of epochs.)


In [3]:
import hashlib, re, urllib.request

BOOKS = {
    "darwin":  dict(gid=1228, title="On the Origin of Species (Charles Darwin, 1859)"),
    "cookery": dict(gid=9101, title="Domestic Cookery (Elizabeth E. Lea, 1853)"),
}
_START = re.compile(r"\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", re.S)
_END = re.compile(r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", re.S)


def strip_gutenberg(raw: str) -> str:
    """Drop the licence header and footer, keeping the book."""
    if (m := _START.search(raw)):
        raw = raw[m.end():]
    if (m := _END.search(raw)):
        raw = raw[:m.start()]
    return raw.strip()


def to_documents(text: str, target_chars: int = 3500, min_chars: int = 1200) -> list[str]:
    """Accumulate blank-line-separated paragraphs into documents of about `target_chars`."""
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    documents, buffer, size = [], [], 0
    for paragraph in paragraphs:
        buffer.append(paragraph)
        size += len(paragraph) + 2
        if size >= target_chars:
            documents.append("\n\n".join(buffer))
            buffer, size = [], 0
    if size >= min_chars:
        documents.append("\n\n".join(buffer))
    return [d for d in documents if len(d) >= min_chars]


def prepare(name: str, hold_out_every: int = 8) -> dict:
    spec = BOOKS[name]
    raw_path = DATA / "raw" / f"pg{spec['gid']}.txt"
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    if not raw_path.exists():
        url = f"https://www.gutenberg.org/cache/epub/{spec['gid']}/pg{spec['gid']}.txt"
        with urllib.request.urlopen(url, timeout=120) as response:
            raw_path.write_bytes(response.read())
    raw = raw_path.read_bytes()
    documents = to_documents(strip_gutenberg(raw.decode("utf-8", errors="replace")))

    train, heldout = DATA / name / "train", DATA / name / "heldout"
    for directory in (train, heldout):
        directory.mkdir(parents=True, exist_ok=True)
        for stale in directory.glob("*.txt"):
            stale.unlink()
    for i, document in enumerate(documents):
        destination = heldout if i % hold_out_every == hold_out_every - 1 else train
        (destination / f"{i:04d}.txt").write_text(document, encoding="utf-8")

    return dict(title=spec["title"], mb=len(raw) / 1e6,
                sha256=hashlib.sha256(raw).hexdigest()[:12],
                train=len(list(train.glob("*.txt"))), heldout=len(list(heldout.glob("*.txt"))))


corpora = {name: prepare(name) for name in BOOKS}
for name, info in corpora.items():
    print(f"{name:<9} {info['title']}\n"
          f"          {info['mb']:.2f} MB downloaded, sha256 {info['sha256']}…, "
          f"{info['train']} training documents + {info['heldout']} held out")

DARWIN_TRAIN, DARWIN_HELD = DATA / "darwin" / "train", DATA / "darwin" / "heldout"
COOKERY_TRAIN, COOKERY_HELD = DATA / "cookery" / "train", DATA / "cookery" / "heldout"

darwin    On the Origin of Species (Charles Darwin, 1859)
          0.97 MB downloaded, sha256 ededa9c0bf87…, 193 training documents + 27 held out
cookery   Domestic Cookery (Elizabeth E. Lea, 1853)
          0.48 MB downloaded, sha256 8ab8a0d20e54…, 105 training documents + 15 held out


In [4]:
for name, path in (("A / darwin", DARWIN_TRAIN), ("B / cookery", COOKERY_TRAIN)):
    sample = sorted(path.glob("*.txt"))[6].read_text(encoding="utf-8")
    print(f"--- {name} " + "-" * (66 - len(name)))
    print(textwrap.fill(" ".join(sample.split())[:520], 88), "…\n")

--- A / darwin --------------------------------------------------------
Sterility has been said to be the bane of horticulture; but on this view we owe
variability to the same cause which produces sterility; and variability is the source of
all the choicest productions of the garden. I may add, that as some organisms will breed
most freely under the most unnatural conditions (for instance, the rabbit and ferret
kept in hutches), showing that their reproductive system has not been thus affected; so
will some animals and plants withstand domestication or cultivation, and vary very
slight …

--- B / cookery -------------------------------------------------------
Celery Sauce. Take a large bunch of celery, cut it fine, and boil it till soft, in a
pint of water; thicken it with butter and flour, and season it with salt, pepper, and
mace. Bacon Dumplings. Cut slices of cooked bacon, and pepper them; roll out crust as
for apple dumplings; slice some potatoes very thin, and put them in the cru

## 3. A workspace

A workspace is the state machine that keeps a chain honest: **which model the next stage
adapts**, **which version of `p(h)` it anchors against**, and **the order the two may be done
in**. Get any of those wrong and the run still completes, still reports a plausible loss, and
quietly measures something else.

`init` records the model rather than loading it, copies the artifact in as `artifacts/v1.pt` so
the workspace owns its own `p(h)`, and picks up the bundled recipe whose `model_id` is this
model. One line on the CLI:

```bash
lfa init lfa_demo/workspace --model Qwen/Qwen3-0.6B --artifact self-generated
```


In [5]:
from lfa import Recipe, Workspace

workspace = Workspace.init(WS, MODEL_ID, artifact=str(artifact_path))
recipe = Recipe.load("qwen3-0.6b")

print(f"model            {workspace.state['model_id']}")
print(f"artifact         v{workspace.state['artifact_version']}  ({workspace.state['artifact_id']})")
print(f"stage            {workspace.state['stage']}")
print("\nthe operating point this recipe carries:")
for field in ("lora_rank", "lora_alpha", "lambda_qkv", "lambda_mlp", "mu", "anchor_end_ratio",
              "n_anchor_samples", "epochs", "learning_rate", "batch_size", "sequence_length",
              "val_fraction", "stage2_lambda_multiplier"):
    print(f"   {field:<26} {getattr(recipe, field)}")

notes = recipe.warnings(rank=recipe.lora_rank, artifact_id=workspace.state["artifact_id"], artifact_meta=params.get("__meta__"))
print(f"\noff-calibration warnings: {notes if notes else 'none — this is the point λ was tuned at'}")

Artifact ~/.cache/lfa/artifacts/Qwen--Qwen3-0.6B-a15c53884fd5-9b4469a9df3a/artifact.pt copied to /tmp/lfa-nb-run/cwd/lfa_demo/workspace/artifacts/v1.pt


Default recipe for this workspace: 'qwen3-0.6b' (it names Qwen/Qwen3-0.6B)


Workspace initialised at /tmp/lfa-nb-run/cwd/lfa_demo/workspace over Qwen/Qwen3-0.6B


model            Qwen/Qwen3-0.6B
artifact         v1  (self-generated:b9cfe7d681e7)
stage            0

the operating point this recipe carries:
   lora_rank                  32
   lora_alpha                 64
   lambda_qkv                 1000000.0
   lambda_mlp                 1000000.0
   mu                         0.05
   anchor_end_ratio           0.1
   n_anchor_samples           16
   epochs                     15
   learning_rate              0.0003
   batch_size                 6
   sequence_length            512
   val_fraction               0.1
   stage2_lambda_multiplier   3.0

off-calibration warnings: none — this is the point λ was tuned at


## 4. Stage 1 — adapt to domain A

One training run. Rank-32 LoRA over the frozen teacher, with three terms in the loss:

```
L  =  L_content                                    cross-entropy on Darwin
   +  λ · E_{h ~ p(h)} ‖f_student(h) − f_teacher(h)‖²    the function anchor
   +  μ · ‖W_student − W_teacher‖²                 the weight backstop
```

The middle term never touches the batch. On every step it draws 16 hidden states per site **from
the artifact**, runs the teacher's sub-module and the student's on them, and penalizes the
difference. That is the whole of the preservation signal, and it is why nothing here needs the
old data.

**Choosing the epochs — do this on your own corpus too.** At the recipe's λ = 1,000,000 this Darwin
text turned late and shallowly over the recipe's 15 epochs: in the run that chose that λ
(`docs/recipes.md`) the held-out perplexity was lowest at epoch 10 (17.62) and ended at 18.09, 2.6 %
above it, a turn for which the trainer advises a re-run at epoch 10 (no 10-epoch run was measured).
A weaker anchor, or a smaller corpus, turns much sooner, and `final_model` is the last epoch by
design. Every run ends by naming its lowest epoch, its final value and the gap between them; when
the curve turned it advises a re-run at that epoch, and when it had not it says more epochs may
lower it (`docs/tuning.md` is the procedure). `EPOCHS = 4` here is the notebook's wall clock, not a
dose read off a curve. A 4-epoch run is not the first four epochs of a 15-epoch one either: the
learning-rate schedule is laid over whatever number of epochs you ask for. Below, the 4-epoch
anchored curve is still falling at its last epoch (20.488, 19.521, 18.859, 18.742), and its last
line says so: this stage stops where the demo's time runs out, not where the text does.

```bash
lfa train --workspace lfa_demo/workspace --corpus lfa_demo/data/darwin/train --epochs 4
```


In [6]:
t0 = time.time()
stage1 = workspace.train(DARWIN_TRAIN, epochs=EPOCHS, device=DEVICE)
stage1_seconds = time.time() - t0

print(f"\nstage {stage1['stage']}: {stage1['n_train_docs']} documents trained, "
      f"{stage1['n_val_docs']} held back for the curve, λ = {stage1['lambda_applied']:,.0f}, "
      f"{stage1_seconds/60:.1f} min")

loader: a document that fits in one chunk is trained whole in every epoch; under keep_short_whole=False it is cut at the epoch's chunk offset like a longer one


Stage 1: Qwen/Qwen3-0.6B on /tmp/lfa-nb-run/cwd/lfa_demo/data/darwin/train (artifact v1, λ_qkv=1000000.0, λ_mlp=1000000.0, μ=0.05, 4 epochs)


Writing the supplement for /tmp/lfa-nb-run/cwd/lfa_demo/data/darwin/train with Qwen/Qwen3-0.6B (173 training documents)


supplement: 16/265 passages -> 95 pairs


supplement: 32/265 passages -> 185 pairs


...

  Training state saved (epoch=4, step=566, checkpoint=final_model)


Training history saved to /tmp/lfa-nb-run/cwd/lfa_demo/workspace/runs/stage1/training_history.json


Held-out perplexity: lowest 18.742 at epoch 4 of 4; final 18.742 (+0.0 % over the lowest).


The held-out curve had not turned by the last epoch, so more epochs may lower it further: the dose can be raised with a re-run at a larger --epochs.



stage 1: 173 documents trained, 20 held back for the curve, λ = 1,000,000, 12.8 min


## 5. The control that makes the point

`evaluate` reads the stage on **both axes at once**, because either alone is meaningless: a run
that reports only the domain has not said what it gave up, and one that reports only WikiText-2
has not said whether it learned anything.

`compare_unanchored=True` adds a third column by training the stage **again** with λ = μ = 0 —
same corpus, same 4 epochs, same learning rate, same seed, same everything except the anchor.
That second full training run is what says what the anchor bought. One caveat travels with it:
4 epochs is the demo's dose, set by its wall clock, and without the anchor the run reaches its own
optimum sooner and trains past it (its output below says so), so the gap is an **upper bound** on
the anchor rather than the anchor alone. Beneath the table `evaluate` first restates this run's own
held-out verdict ("This run's held-out curve: …"; here, that the anchored curve had not turned),
then prints the same caveat and the commands that train the control at its own best epoch in a
fresh workspace.

The domain axis is Darwin's `heldout/` directory, passed explicitly. (`evaluate` logs "a fit, not
a held-out measurement" whenever a corpus is named, because a named corpus is not split — these
documents were never in the training directory at all, and naming it is what makes it the *same*
text as every later domain-A read.)

```bash
lfa evaluate --workspace lfa_demo/workspace --corpus lfa_demo/data/darwin/heldout \
             --compare-unanchored --n-windows 200
```


In [7]:
t0 = time.time()
scores1 = workspace.evaluate(corpus=DARWIN_HELD, compare_unanchored=True,
                             n_windows=N_WINDOWS, device=DEVICE)
control1_seconds = time.time() - t0
print("\n" + scores1["table"])

Evaluating stage 1 on /tmp/lfa-nb-run/cwd/lfa_demo/data/darwin/heldout (64 chunks, trained on -- a fit, not a held-out measurement)


The run read: /tmp/lfa-nb-run/cwd/lfa_demo/workspace/runs/stage1 (λ 1e+06 (the recipe's), μ 0.05 (the recipe's), 4 epochs)


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at ~/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


...

Found the latest cached dataset configuration 'wikitext-2-raw-v1' at ~/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at ~/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


WikiText-2 perplexity 26.68 (loss 3.2840 over 102400 tokens, window 2048 / stride 512)


Domain perplexity 18.82 (loss 2.9347 over 26147 tokens in 64 chunks)



| metric               | before | after |     Δ% | unanchored |     Δ% |
| -------------------- | -----: | ----: | -----: | ---------: | -----: |
| general (WikiText-2) |  17.80 | 16.08 |  -9.7% |      26.68 | +49.9% |
| domain               |  30.12 | 19.30 | -35.9% |      18.82 | -37.5% |

This run's held-out curve: lowest 18.742 at epoch 4 of 4; final 18.742 (+0.0 % over the lowest). The
held-out curve had not turned by the last epoch, so more epochs may lower it further: the dose can
be raised with a re-run at a larger --epochs.

The unanchored control's held-out perplexity was lowest at epoch 2 (14.85) and ended at 18.23 (epoch
4, +22.8 %): it trained at this run's dose, past its own best, so part of the unanchored column's
gap is dose, not the anchor. To read the anchor alone, train the control at its own best epoch in a
fresh workspace and set its `after` column beside this run's. `lfa init` ends with its own Next: and
Then: suggestions; none of them applies to this lambda-0 co

In [8]:
before, after, unanchored = scores1["before"], scores1["after"], scores1["unanchored"]
print(f"""
Read it as two questions.

  Did it learn Darwin?   {before['domain']:.2f} -> {after['domain']:.2f}
                         ({(after['domain']/before['domain'] - 1) * 100:+.1f}% held-out domain perplexity)

  What did that cost?    WikiText-2 {before['general']:.2f} -> {after['general']:.2f}
                         ({(after['general']/before['general'] - 1) * 100:+.1f}%)

  Without the anchor:    Darwin      {before['domain']:.2f} -> {unanchored['domain']:.2f}   ({(unanchored['domain']/before['domain'] - 1) * 100:+.1f}%)
                         WikiText-2  {before['general']:.2f} -> {unanchored['general']:.2f}   ({(unanchored['general']/before['general'] - 1) * 100:+.1f}%)
""")


Read it as two questions.

  Did it learn Darwin?   30.12 -> 19.30
                         (-35.9% held-out domain perplexity)

  What did that cost?    WikiText-2 17.80 -> 16.08
                         (-9.7%)

  Without the anchor:    Darwin      30.12 -> 18.82   (-37.5%)
                         WikiText-2  17.80 -> 26.68   (+49.9%)



## 6. `extend` — folding A in, so that B never has to see it

A second domain is not just a second training run. Stage 2 has to adapt the model that already
carries A, and it has to anchor against a `p(h)` that *describes* that model — otherwise it
preserves a function the model no longer computes. `extend` does both, and the workspace refuses
to train a different corpus until it has:

* **Merge.** Stage 1's adapter is merged into the model it trained over, giving
  `models/stage1_fused` — a plain checkpoint, and what stage 2 adapts.
* **Extend `p(h)`.** Domain A is run through the *fused* model, its activations are fitted as a
  small mixture **in the base artifact's own PCA basis**, and the two mixtures are merged as a
  sample-weighted union:

  ```
  p(h) = (1 − a) · p_base(h) + a · p_domain(h),      a = n_domain / (n_base + n_domain)
  ```

The union is exact, so nothing is refitted: each round reads only the **new** domain and adds a
fixed number of components per site, and the cost per round is flat in the number of rounds.

The output is `artifacts/v2.pt`, still a set of per-site tensors. **No Darwin text is stored
anywhere in the workspace from here on** — the second cell below checks that by searching every
byte the workspace holds.

`--need` is the memory bill: one `need x width` float32 buffer per site, on the host, for all 84
sites at once — about 9 GB of RAM at the 20,000 used here, 18 GB at the 40,000 default.

```bash
lfa extend --workspace lfa_demo/workspace --need 20000 --k-domain 4
```


In [9]:
t0 = time.time()
artifact_v2 = workspace.extend(need=EXTEND_NEED, k_domain=EXTEND_K, device=DEVICE)
extend_seconds = time.time() - t0
print(f"\n{artifact_v2}  ({artifact_v2.stat().st_size / 1e6:.1f} MB, {extend_seconds/60:.1f} min)")
print(f"current model    {workspace.state['current_model']}")
print(f"current artifact v{workspace.state['artifact_version']}")

Fusing adapter /tmp/lfa-nb-run/cwd/lfa_demo/workspace/runs/stage1/final_model into Qwen/Qwen3-0.6B


Fused model and tokenizer written to /tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused


The tokenizer you are loading from '/tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Collected 20000 activations at each of 84 sites from 52 chunks


Fitting the domain mixture: site 1/84 (0_pre_o)


...

Fitting the domain mixture: site 80/84 (26_pre_mlp)


Fitting the domain mixture: site 84/84 (28_pre_lm_head)


Extended 84 sites with K 4-4 new components from 20000 activations (domain weight share 0.0321); wrote version 2


Stage 1 folded in: model /tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused, artifact v2



/tmp/lfa-nb-run/cwd/lfa_demo/workspace/artifacts/v2.pt  (110.5 MB, 1.0 min)
current model    /tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused
current artifact v2


In [10]:
v1, v2 = load_artifact(artifact_path), load_artifact(artifact_v2)
site = "12_pre_mlp"
basis_shift = (v1[site]["pca_components"].float()
               - v2[site]["pca_components"].float()).abs().max().item()

print(f"{site}: mixture components  v1 {v1[site]['gmm_n_components']}"
      f"  ->  v2 {v2[site]['gmm_n_components']}   (+{EXTEND_K} for domain A)")
print(f"          basis: largest entry-wise change {basis_shift:.2e} — the domain mixture is")
print(f"                 fitted IN the base basis, so this is int8 round-trip noise, not a refit")
print(f"          value types: {sorted({type(x).__name__ for x in v2[site].values()})}")


# The whole point of the method, stated as a filesystem fact: search every byte the workspace
# holds — adapters, optimizer state, the fused model, both artifacts — for a phrase the domain-A
# corpus is full of.
def contains(path, needle, chunk=1 << 22):
    with path.open("rb") as handle:
        tail = b""
        while (block := handle.read(chunk)):
            if needle in tail + block:
                return True
            tail = block[-len(needle):]
    return False


NEEDLE = b"natural selection"
files = [p for p in WS.rglob("*") if p.is_file()]
carriers = [p.relative_to(WS) for p in files if contains(p, NEEDLE)]
occurrences = sum(doc.read_bytes().count(NEEDLE) for doc in DARWIN_TRAIN.glob("*.txt"))

print(f"\n{NEEDLE.decode()!r} appears {occurrences} times in the domain-A training corpus.")
print(f"Files in the workspace ({len(files)} files, "
      f"{sum(p.stat().st_size for p in files) / 1e9:.2f} GB) that contain it: {len(carriers)}")

12_pre_mlp: mixture components  v1 32  ->  v2 36   (+4 for domain A)
          basis: largest entry-wise change 0.00e+00 — the domain mixture is
                 fitted IN the base basis, so this is int8 round-trip noise, not a refit
          value types: ['Tensor', 'float', 'int', 'str']



'natural selection' appears 186 times in the domain-A training corpus.
Files in the workspace (29 files, 1.91 GB) that contain it: 1


## 7. Stage 2 — adapt to domain B

Same command as stage 1. Three things differ, and the workspace supplies all three:

* it adapts `models/stage1_fused`, not the base model;
* it anchors against `artifacts/v2.pt`, which describes *that* model on *both* domains;
* λ is multiplied by the recipe's `stage2_lambda_multiplier` (3.0), because a later stage holds
  more than the base model's functions in place. That multiplier is a **level, not a compounding
  factor** — stage 3 anchors at the same multiple, not its square — and 3.0 was measured on this
  same pair of books at the recipe's 15 epochs, where it beat 1.0 on cookery, on Darwin kept and
  on WikiText-2 (`docs/recipes.md`).

Nothing from Darwin is read: no buffer, no rehearsal set, no replay mixture.

```bash
lfa train --workspace lfa_demo/workspace --corpus lfa_demo/data/cookery/train --epochs 4
```


In [11]:
t0 = time.time()
stage2 = workspace.train(COOKERY_TRAIN, epochs=EPOCHS, device=DEVICE)
stage2_seconds = time.time() - t0

print(f"\nstage {stage2['stage']}: {stage2['n_train_docs']} documents, "
      f"λ = {stage2['lambda_applied']:,.0f} "
      f"(= {recipe.lambda_qkv:,.0f} × {recipe.stage2_lambda_multiplier}), "
      f"anchored against artifact v{stage2['artifact_version']}, {stage2_seconds/60:.1f} min")
print(f"          starting model: {stage2['base_model']}")

loader: a document that fits in one chunk is trained whole in every epoch; under keep_short_whole=False it is cut at the epoch's chunk offset like a longer one


Stage 2: /tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused on /tmp/lfa-nb-run/cwd/lfa_demo/data/cookery/train (artifact v2, λ_qkv=3000000.0, λ_mlp=3000000.0, μ=0.05, 4 epochs)


Writing the supplement for /tmp/lfa-nb-run/cwd/lfa_demo/data/cookery/train with /tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused (94 training documents)


The tokenizer you are loading from '/tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


supplement: 16/106 passages -> 94 pairs


...

  Training state saved (epoch=4, step=297, checkpoint=final_model)


Training history saved to /tmp/lfa-nb-run/cwd/lfa_demo/workspace/runs/stage2/training_history.json


Held-out perplexity: lowest 16.158 at epoch 4 of 4; final 16.158 (+0.0 % over the lowest).


The held-out curve had not turned by the last epoch, so more epochs may lower it further: the dose can be raised with a re-run at a larger --epochs.



stage 2: 94 documents, λ = 3,000,000 (= 1,000,000 × 3.0), anchored against artifact v2, 7.7 min
          starting model: /tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused


## 8. The contrast again, and the question the whole notebook is about

First the same two-axis read, with its own unanchored control: stage 2 re-run from the same
starting model with λ = μ = 0. That table's "before" column is the model **after domain A**, so it
reads stage 2 in isolation and does not yet answer the question the method exists for, which is
about A.

So, after it: five models, three axes, one table. The general axis is WikiText-2, which no stage
trained on; the two domain axes are the `heldout/` documents, which no stage trained on either.


In [12]:
t0 = time.time()
scores2 = workspace.evaluate(corpus=COOKERY_HELD, compare_unanchored=True,
                             n_windows=N_WINDOWS, device=DEVICE)
control2_seconds = time.time() - t0
print("\n" + scores2["table"])

The tokenizer you are loading from '/tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Evaluating stage 2 on /tmp/lfa-nb-run/cwd/lfa_demo/data/cookery/heldout (33 chunks, trained on -- a fit, not a held-out measurement)


The run read: /tmp/lfa-nb-run/cwd/lfa_demo/workspace/runs/stage2 (λ 3e+06 (the recipe's), μ 0.05 (the recipe's), 4 epochs)


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


...

Found the latest cached dataset configuration 'wikitext-2-raw-v1' at ~/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at ~/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


WikiText-2 perplexity 25.15 (loss 3.2247 over 102400 tokens, window 2048 / stride 512)


Domain perplexity 14.33 (loss 2.6626 over 14949 tokens in 33 chunks)



| metric               | before | after |     Δ% | unanchored |     Δ% |
| -------------------- | -----: | ----: | -----: | ---------: | -----: |
| general (WikiText-2) |  16.05 | 15.83 |  -1.4% |      25.15 | +56.7% |
| domain               |  21.69 | 16.99 | -21.6% |      14.33 | -33.9% |

This run's held-out curve: lowest 16.158 at epoch 4 of 4; final 16.158 (+0.0 % over the lowest). The
held-out curve had not turned by the last epoch, so more epochs may lower it further: the dose can
be raised with a re-run at a larger --epochs.

The unanchored control's held-out perplexity was lowest at epoch 2 (11.76) and ended at 13.33 (epoch
4, +13.4 %): it trained at this run's dose, past its own best, so part of the unanchored column's
gap is dose, not the anchor. To read the anchor alone, train the control at its own best epoch in a
fresh workspace and set its `after` column beside this run's. `lfa init` ends with its own Next: and
Then: suggestions (and, from a later stage's model, a warni

In [13]:
import gc
from peft import PeftModel
from lfa.corpus import load_corpus
from lfa.evaluate import domain_perplexity, wikitext2_perplexity
from lfa.models import load_teacher, load_tokenizer

tokenizer = load_tokenizer(MODEL_ID)


def held_out(path):
    corpus, _ = load_corpus(path, tokenizer, max_length=recipe.sequence_length,
                            val_fraction=0.0, seed=recipe.seed,
                            keep_short_whole=recipe.keep_short_whole)
    return corpus


DOMAINS = {"A · Darwin": held_out(DARWIN_HELD), "B · cookery": held_out(COOKERY_HELD)}
STAGE1_FUSED = WS / "models" / "stage1_fused"

MODELS = {
    "base Qwen3-0.6B":            (MODEL_ID, None),
    "after A — anchored":         (MODEL_ID, WS / "runs" / "stage1" / "final_model"),
    "after A — NO anchor":        (MODEL_ID, WS / "runs" / "stage1_unanchored" / "final_model"),
    "after A+B — anchored":       (STAGE1_FUSED, WS / "runs" / "stage2" / "final_model"),
    "after A+B — NO anchor on B": (STAGE1_FUSED, WS / "runs" / "stage2_unanchored" / "final_model"),
}


def score(base, adapter):
    model = load_teacher(str(base), device=DEVICE, dtype=torch.bfloat16)
    if adapter is not None:
        model = PeftModel.from_pretrained(model, str(adapter))
    try:
        row = {"WikiText-2": wikitext2_perplexity(model, tokenizer, n_windows=N_WINDOWS,
                                                  device=DEVICE)}
        row.update({name: domain_perplexity(model, tokenizer, corpus, device=DEVICE)
                    for name, corpus in DOMAINS.items()})
        return row
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()


logging.getLogger("lfa.evaluate").setLevel(logging.WARNING)   # the table is the output here
t0 = time.time()
table = {label: score(*spec) for label, spec in MODELS.items()}
final_table_seconds = time.time() - t0

Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at ~/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at ~/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


...

Found the latest cached dataset configuration 'wikitext-2-raw-v1' at ~/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at ~/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at ~/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


In [14]:
columns = ["WikiText-2", "A · Darwin", "B · cookery"]
width = max(len(label) for label in table)
base_row = table["base Qwen3-0.6B"]

header = f"| {'model':<{width}} | " + " | ".join(f"{c:>18}" for c in columns) + " |"
print(header)
print(f"| {'-' * width} | " + " | ".join("-" * 18 for _ in columns) + " |")
for label, row in table.items():
    cells = []
    for column in columns:
        delta = (row[column] / base_row[column] - 1) * 100
        cells.append(f"{row[column]:8.2f}" + ("         " if label.startswith("base")
                                              else f" ({delta:+5.1f}%)"))
    print(f"| {label:<{width}} | " + " | ".join(f"{c:>18}" for c in cells) + " |")
print("\nperplexity, lower is better; % is against the base model on the same text")

| model                      |         WikiText-2 |         A · Darwin |        B · cookery |
| -------------------------- | ------------------ | ------------------ | ------------------ |
| base Qwen3-0.6B            |     17.80          |     30.12          |     26.51          |
| after A — anchored         |     16.08 ( -9.7%) |     19.30 (-35.9%) |     21.70 (-18.2%) |
| after A — NO anchor        |     26.68 (+49.9%) |     18.82 (-37.5%) |     38.79 (+46.3%) |
| after A+B — anchored       |     15.83 (-11.1%) |     19.97 (-33.7%) |     16.99 (-35.9%) |
| after A+B — NO anchor on B |     25.15 (+41.3%) |     37.67 (+25.0%) |     14.33 (-45.9%) |

perplexity, lower is better; % is against the base model on the same text


### How to read that table

Four comparisons, in the order they answer the notebook's question.

1. **Row 2 against row 1** — did the anchored stage learn Darwin, and what did WikiText-2 do?
2. **Row 3 against row 2** — the control. Same corpus, same steps, anchor off. In this recording
   the unanchored run ends slightly ahead on Darwin (18.82 against 19.30) and far behind on
   WikiText-2 (26.68 against 16.08), and at 4 epochs it has trained past its own optimum (the
   trainer says so in section 5's output). Read that gap as an **upper bound** on what the anchor
   bought, not as the anchor alone: both controls are trained at the demo's number of epochs, not
   their own. `what_the_anchor_does.ipynb` re-runs each control at its own dose, and there it
   reaches the new domain *harder* than the anchored run and pays for it on the general column.
3. **Row 4's `A · Darwin` cell against row 2's** — the chain question. Stage 2 trained on nothing
   but recipes, and read no Darwin text at all. How much of A survived?
4. **Row 5's `A · Darwin` cell against row 4's** — the same stage without the anchor. Both start
   from the same model that already knows A; only one of them is holding it.

Two warnings from `docs/concepts.md` are why nothing here is read on one axis alone: a general
perplexity *below* the base model's is not by itself evidence that anything was preserved, and
over-anchoring makes the general axis look its best while the domain quietly collapses.


In [15]:
a_after_a = table["after A — anchored"]["A · Darwin"]
a_after_ab = table["after A+B — anchored"]["A · Darwin"]
a_after_ab_free = table["after A+B — NO anchor on B"]["A · Darwin"]
g_anchored = table["after A — anchored"]["WikiText-2"]
g_free = table["after A — NO anchor"]["WikiText-2"]

print(f"""
1. stage 1, anchored     Darwin {base_row['A · Darwin']:.2f} -> {a_after_a:.2f}    WikiText-2 {base_row['WikiText-2']:.2f} -> {g_anchored:.2f}
2. stage 1, no anchor    Darwin {base_row['A · Darwin']:.2f} -> {table['after A — NO anchor']['A · Darwin']:.2f}    WikiText-2 {base_row['WikiText-2']:.2f} -> {g_free:.2f}
   what the anchor bought on the general axis: {(g_free / g_anchored - 1) * 100:+.1f}% worse without it

3. domain A across stage 2, anchored:     {a_after_a:.2f} -> {a_after_ab:.2f}   ({(a_after_ab / a_after_a - 1) * 100:+.1f}%)
4. domain A across stage 2, anchor off:   {a_after_a:.2f} -> {a_after_ab_free:.2f}   ({(a_after_ab_free / a_after_a - 1) * 100:+.1f}%)
   stage 2 read 0 words of Darwin in either case.
""")


1. stage 1, anchored     Darwin 30.12 -> 19.30    WikiText-2 17.80 -> 16.08
2. stage 1, no anchor    Darwin 30.12 -> 18.82    WikiText-2 17.80 -> 26.68
   what the anchor bought on the general axis: +66.0% worse without it

3. domain A across stage 2, anchored:     19.30 -> 19.97   (+3.5%)
4. domain A across stage 2, anchor off:   19.30 -> 37.67   (+95.2%)
   stage 2 read 0 words of Darwin in either case.



## 9. Export: a plain checkpoint

`fuse` merges the last stage's adapter into the model it trained over and writes an ordinary
Hugging Face checkpoint — no PEFT wrapper, no `lfa` at load time. It loads with
`AutoModelForCausalLM.from_pretrained` like any other model.

```bash
lfa fuse --workspace lfa_demo/workspace
```


In [16]:
exported = workspace.fuse()
print(exported)
print(sorted(p.name for p in exported.iterdir()))

from transformers import AutoModelForCausalLM, AutoTokenizer

merged = AutoModelForCausalLM.from_pretrained(str(exported), dtype=torch.bfloat16).to(DEVICE)
merged_tokenizer = AutoTokenizer.from_pretrained(str(exported))

PROMPTS = [
    "The several difficulties which have been discussed, namely, that",   # domain A
    "To make a plain beef soup, take",                                    # domain B
]
for prompt in PROMPTS:
    ids = merged_tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = merged.generate(**ids, max_new_tokens=60, do_sample=False,
                              pad_token_id=merged_tokenizer.eos_token_id)
    print("· " + textwrap.fill(merged_tokenizer.decode(out[0], skip_special_tokens=True),
                               88, subsequent_indent="  ") + "\n")

del merged
gc.collect()
torch.cuda.empty_cache()

Fusing adapter /tmp/lfa-nb-run/cwd/lfa_demo/workspace/runs/stage2/final_model into /tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused


The tokenizer you are loading from '/tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage1_fused' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Fused model and tokenizer written to /tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage2_fused_export


/tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage2_fused_export
['added_tokens.json', 'chat_template.jinja', 'config.json', 'generation_config.json', 'merges.txt', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']


The tokenizer you are loading from '/tmp/lfa-nb-run/cwd/lfa_demo/workspace/models/stage2_fused_export' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


· The several difficulties which have been discussed, namely, that the sun is not a star,
  and that the sun is not a planet, and that the sun is not a star, and that the sun is
  not a planet, and that the sun is not a star, and that the sun is not a planet, and
  that the sun



· To make a plain beef soup, take a pint of beef, cut it up, and put it in a pot, with a
  pint of water, and a pint of milk, and a thousandth of a pound of salt, and a little
  of sugar, and a little of pepper, and boil it for half



Those two continuations are **qualitative only** — a greedy decode from one small model, and
nothing in this package scores generated text. They are here to show one set of weights
taking a prompt from each domain. The cookery prompt is continued in the cookbook's manner ("cut
it up, and put it in a pot, with a pint of water"); the Darwin prompt is not continued in
Darwin's, and shows what plain greedy decoding does to a model this size: it repeats a clause
("the sun is not a star, and that the sun is not a planet") until the 60 tokens run out. The
companion notebook's decode adds two repetition controls for that reason.

## 10. The whole chain, and what it cost


In [17]:
import json

for entry in json.load((WS / "history.json").open()):
    perplexity = entry.get("perplexity") or {}
    after = (perplexity.get("after") or {})
    print(f"stage {entry['stage']}  {Path(entry['corpus']).parent.name:<9} "
          f"λ={entry['lambda_applied']:>9,.0f}  artifact v{entry['artifact_version']}  "
          f"{entry['n_train_docs']:>3} docs  "
          f"domain {after.get('domain', float('nan')):.2f}  "
          f"general {after.get('general', float('nan')):.2f}")

print(f"""
wall clock
  stage 1 (anchored)                {stage1_seconds / 60:5.1f} min
  stage 1 evaluate + control        {control1_seconds / 60:5.1f} min
  extend (fuse + fit + merge p(h))  {extend_seconds / 60:5.1f} min
  stage 2 (anchored)                {stage2_seconds / 60:5.1f} min
  stage 2 evaluate + control        {control2_seconds / 60:5.1f} min
  the five-model table              {final_table_seconds / 60:5.1f} min
  ------------------------------------------
  notebook, end to end              {(time.time() - NOTEBOOK_STARTED) / 60:5.1f} min
""")

stage 1  darwin    λ=1,000,000  artifact v1  173 docs  domain 19.30  general 16.08
stage 2  cookery   λ=3,000,000  artifact v2   94 docs  domain 16.99  general 15.83

wall clock
  stage 1 (anchored)                 12.8 min
  stage 1 evaluate + control          7.2 min
  extend (fuse + fit + merge p(h))    1.0 min
  stage 2 (anchored)                  7.7 min
  stage 2 evaluate + control          4.3 min
  the five-model table                2.3 min
  ------------------------------------------
  notebook, end to end               35.5 min



## What this showed, and what it did not

**Showed.** Two domains adapted in sequence on one 0.6B model, the second stage reading none of
the first domain's text and the workspace storing none of it, both stages measured on two axes
against a control that differs only in λ and μ.

**Did not show.** Anything statistical. One seed, one run per cell, a quarter of the recipe's
epochs — so a few percent in any cell is not distinguishable from run-to-run variation. It
reproduces no published number.

**To run this on your own corpus**: replace section 2 with a directory of `.txt` files, let the
held-out curve pick the epochs the way section 4 describes, and re-tune λ if you change the rank,
the artifact or the corpus — it is coupled to all three, and a ported λ measures the port rather
than the method. `docs/tuning.md` is how, on your own corpus.

**If you want to see what the anchor does to the model's answers rather than to its
perplexities**, open [`what_the_anchor_does.ipynb`](what_the_anchor_does.ipynb). It continues
from the `lfa_demo/` workspace this notebook just built: the same three models put to fixed
probes, and each control re-run at its own best number of epochs, which splits the gaps above
into what is dose and what is anchor.

### Where to go next

| | |
|---|---|
| `docs/tuning.md` | the epochs and λ on your own corpus |
| `docs/concepts.md` | what the anchor does, what λ and μ are, how a run is read |
| `docs/recipes.md` | the shipped operating point field by field, and its couplings |
| `docs/multi-domain-chains.md` | a third domain, `lfa chain`, and what the paper found after one |
| `docs/faq.md` | GPU memory, full weights, reading the general axis |
| `examples/quickstart.py` | the single-domain flow as a script |
